In [ ]:
import pandas as pd
import torch
from transformers import EncoderDecoderModel, AutoTokenizer
from tqdm import tqdm
import numpy as np
from difflib import SequenceMatcher
import os

# 평가 metrics

In [ ]:
def calculate_char_accuracy(pred, target):
    """문자 단위 정확도"""
    if len(target) == 0:
        return 0.0
    correct = sum(1 for p, t in zip(pred, target) if p == t)
    return correct / max(len(pred), len(target))

def calculate_word_accuracy(pred, target):
    """단어 단위 정확도"""
    pred_words = pred.split()
    target_words = target.split()
    if len(target_words) == 0:
        return 0.0
    correct = sum(1 for p, t in zip(pred_words, target_words) if p == t)
    return correct / max(len(pred_words), len(target_words))

def calculate_similarity(pred, target):
    """문자열 유사도 (SequenceMatcher)"""
    return SequenceMatcher(None, pred, target).ratio()

# data load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("데이터 로딩 중...")
train = pd.read_csv('/content/drive/MyDrive/data/open/train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('/content/drive/MyDrive/data/open/test.csv', encoding = 'utf-8-sig')

데이터 로딩 중...


In [ ]:
# 평가용 샘플 (전체 사용 시 시간이 오래 걸릴 수 있음)
train_samples = train[:200]  # Few-shot 예시용
eval_samples = train[200:300]  # 평가용 (train의 일부를 평가용으로)

In [ ]:
# Few-shot 예시 생성
samples = []
for i in range(10):
    sample = f"input : {train_samples['input'].iloc[i]} \n output : {train_samples['output'].iloc[i]}"
    samples.append(sample)

In [ ]:
# 평가할 모델 리스트
encoder_models = [
    "kakaobank/kf-deberta-base",
    "monologg/koelectra-small-v2-discriminator",
    "klue/roberta-small",
    "beomi/KcELECTRA-base",
]

In [ ]:
# 결과 파일 경로
results_file_path = '/content/drive/MyDrive/data/open/model_evaluation_results.csv'

# 기존 결과 파일 로드
already_evaluated = []
if os.path.exists(results_file_path):
    print("\n기존 평가 결과 파일을 찾았습니다.")
    existing_results = pd.read_csv(results_file_path, encoding='utf-8-sig')
    already_evaluated = existing_results['model_name'].tolist()
    print(f"이미 평가된 모델: {already_evaluated}")

def save_result_immediately(result_dict, results_file_path):
    """각 모델 평가 후 즉시 결과를 파일에 저장"""
    if os.path.exists(results_file_path):
        existing = pd.read_csv(results_file_path, encoding='utf-8-sig')
        existing = existing[existing['model_name'] != result_dict['model_name']]
        updated = pd.concat([existing, pd.DataFrame([result_dict])], ignore_index=True)
    else:
        updated = pd.DataFrame([result_dict])

    updated = updated.sort_values('avg_score', ascending=False).reset_index(drop=True)
    updated.to_csv(results_file_path, index=False, encoding='utf-8-sig')
    print(f"✓ 결과가 저장되었습니다: {results_file_path}")


기존 평가 결과 파일을 찾았습니다.
이미 평가된 모델: ['beomi/gemma-ko-2b', 'paust/pko-t5-base', 'lcw99/t5-base-korean-text-summary', 'psyche/KoT5-summarization', 'cosmoquester/bart-ko-small', 'eenzeenee/t5-base-korean-summarization', 'gogamza/kobart-summarization', 'Jo-2j/Dacon-Encrypted', 'gogamza/kobart-base-v2', 'facebook/mbart-large-50', 'KETI-AIR/ke-t5-base', 'cchyun/nmt-koen-t5-small', 'lcw99/t5-base-korean-grammar-correction', 'MLP-KTLim/llama-3-Korean-Bllossom-AICA-3B', nan, nan, nan, nan, nan]


In [ ]:
results = []

for encoder_model_name in encoder_models:
    # EncoderDecoder 모델명 생성
    model_name = f"{encoder_model_name}-encoder-decoder"

    if model_name in already_evaluated:
        print(f"\n{'='*60}")
        print(f"모델 '{model_name}'은 이미 평가되었습니다. 건너뜁니다.")
        print(f"{'='*60}\n")
        continue

    print(f"\n{'='*60}")
    print(f"모델 평가 중: {model_name}")
    print(f"인코더: {encoder_model_name}")
    print(f"{'='*60}\n")

    try:
        # 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(encoder_model_name)

        # pad_token 설정
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token or '[PAD]'
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})

        if tokenizer.bos_token is None:
            if tokenizer.cls_token:
                tokenizer.bos_token = tokenizer.cls_token
            else:
                tokenizer.add_special_tokens({'bos_token': '[BOS]'})

        print(f"특수 토큰 설정:")
        print(f"  - pad_token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")
        print(f"  - bos_token: {tokenizer.bos_token} (id: {tokenizer.bos_token_id})")
        print(f"  - eos_token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")

        print("EncoderDecoder 모델 생성 중...")
        # EncoderDecoder 모델 생성 (같은 모델을 encoder와 decoder로 사용)
        model = EncoderDecoderModel.from_encoder_decoder_pretrained(
            encoder_model_name,
            encoder_model_name,
            tie_encoder_decoder=True  # 파라미터 공유로 메모리 절약
        )

        # 토크나이저 크기 조정 (새로운 특수 토큰 추가된 경우)
        model.encoder.resize_token_embeddings(len(tokenizer))
        model.decoder.resize_token_embeddings(len(tokenizer))

        # 특수 토큰 설정 (필수!)
        model.config.decoder_start_token_id = tokenizer.bos_token_id
        model.config.bos_token_id = tokenizer.bos_token_id
        model.config.eos_token_id = tokenizer.eos_token_id
        model.config.pad_token_id = tokenizer.pad_token_id

        # 추가 설정
        model.config.vocab_size = model.config.encoder.vocab_size
        model.config.max_length = 128
        model.config.min_length = 5
        model.config.no_repeat_ngram_size = 3

        print(f"모델 설정:")
        print(f"  - decoder_start_token_id: {model.config.decoder_start_token_id}")
        print(f"  - bos_token_id: {model.config.bos_token_id}")
        print(f"  - eos_token_id: {model.config.eos_token_id}")
        print(f"  - pad_token_id: {model.config.pad_token_id}")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)

        # Fine-tuning (간단한 학습)
        print("모델 Fine-tuning 중... (간단한 학습)")
        model.train()
        optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

        # 일부 데이터로만 빠르게 학습
        num_train_samples = min(100, len(train_samples))  # 100개만 사용
        for epoch in range(1):  # 1 epoch만
            for idx in tqdm(range(num_train_samples), desc=f"Training"):
                input_text = train_samples['input'].iloc[idx]
                target_text = train_samples['output'].iloc[idx]

                # 입력과 타겟 인코딩
                inputs = tokenizer(
                    input_text,
                    return_tensors="pt",
                    max_length=128,
                    truncation=True,
                    padding="max_length"
                ).to(device)

                labels = tokenizer(
                    target_text,
                    return_tensors="pt",
                    max_length=128,
                    truncation=True,
                    padding="max_length"
                ).input_ids.to(device)

                # -100으로 패딩 마스킹
                labels[labels == tokenizer.pad_token_id] = -100

                # Forward pass
                outputs = model(
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        print("Fine-tuning 완료! 평가 시작...")
        model.eval()

        # 평가 메트릭 저장
        char_accuracies = []
        word_accuracies = []
        similarities = []

        # 각 샘플에 대해 추론
        for idx, row in tqdm(eval_samples.iterrows(), total=len(eval_samples), desc=f"Evaluating {model_name}"):
            query = row['input']
            target = row['output']

            try:
                inputs = tokenizer(
                    query,
                    return_tensors="pt",
                    max_length=128,
                    truncation=True
                ).to(device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_length=128,
                        num_beams=4,
                        early_stopping=True,
                        decoder_start_token_id=model.config.decoder_start_token_id,
                        bos_token_id=model.config.bos_token_id,
                        eos_token_id=model.config.eos_token_id,
                        pad_token_id=model.config.pad_token_id
                    )

                result = tokenizer.decode(outputs[0], skip_special_tokens=True)

                # 메트릭 계산
                char_acc = calculate_char_accuracy(result, target)
                word_acc = calculate_word_accuracy(result, target)
                sim = calculate_similarity(result, target)

                char_accuracies.append(char_acc)
                word_accuracies.append(word_acc)
                similarities.append(sim)

            except Exception as e:
                print(f"Error processing sample {idx}: {str(e)}")
                continue

        # 평균 메트릭 계산
        avg_char_acc = np.mean(char_accuracies) if char_accuracies else 0.0
        avg_word_acc = np.mean(word_accuracies) if word_accuracies else 0.0
        avg_sim = np.mean(similarities) if similarities else 0.0

        # 결과 저장
        result_dict = {
            "model_name": model_name,
            "char_accuracy": avg_char_acc,
            "word_accuracy": avg_word_acc,
            "similarity": avg_sim,
            "avg_score": (avg_char_acc + avg_word_acc + avg_sim) / 3,
            "model_type": "EncoderDecoder",
            "encoder": encoder_model_name
        }
        results.append(result_dict)

        # 즉시 파일에 저장
        save_result_immediately(result_dict, results_file_path)

        print(f"\n[{model_name}] 평가 결과:")
        print(f"  - 문자 정확도: {avg_char_acc:.4f}")
        print(f"  - 단어 정확도: {avg_word_acc:.4f}")
        print(f"  - 유사도: {avg_sim:.4f}")
        print(f"  - 평균 점수: {result_dict['avg_score']:.4f}")

        # 메모리 및 디스크 정리
        del model
        del tokenizer
        del optimizer
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error loading model {encoder_model_name}: {str(e)}")
        import traceback
        traceback.print_exc()

        error_result = {
            "model_name": model_name,
            "char_accuracy": 0.0,
            "word_accuracy": 0.0,
            "similarity": 0.0,
            "avg_score": 0.0,
            "model_type": "EncoderDecoder",
            "encoder": encoder_model_name,
            "error": str(e)
        }
        results.append(error_result)
        save_result_immediately(error_result, results_file_path)
        continue


모델 평가 중: kakaobank/kf-deberta-base-encoder-decoder
인코더: kakaobank/kf-deberta-base

특수 토큰 설정:
  - pad_token: [PAD] (id: 0)
  - bos_token: [CLS] (id: 2)
  - eos_token: [SEP] (id: 3)
EncoderDecoder 모델 생성 중...
Error loading model kakaobank/kf-deberta-base: Unrecognized configuration class <class 'transformers.models.deberta_v2.configuration_deberta_v2.DebertaV2Config'> for this kind of AutoModel: AutoModelForCausalLM.
Model type should be one of ArceeConfig, AriaTextConfig, BambaConfig, BartConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BitNetConfig, BlenderbotConfig, BlenderbotSmallConfig, BloomConfig, CamembertConfig, LlamaConfig, CodeGenConfig, CohereConfig, Cohere2Config, CpmAntConfig, CTRLConfig, Data2VecTextConfig, DbrxConfig, DeepseekV3Config, DiffLlamaConfig, Dots1Config, ElectraConfig, Emu3Config, ErnieConfig, FalconConfig, FalconH1Config, FalconMambaConfig, FuyuConfig, GemmaConfig, Gemma2Config, Gemma3Config, Gemma3TextConfig, Gemma3

Traceback (most recent call last):
  File "/tmp/ipython-input-14-3457060518.py", line 40, in <cell line: 0>
    model = EncoderDecoderModel.from_encoder_decoder_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py", line 442, in from_encoder_decoder_pretrained
    decoder = AutoModelForCausalLM.from_pretrained(decoder_pretrained_model_name_or_path, **kwargs_decoder)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py", line 603, in from_pretrained
    raise ValueError(
ValueError: Unrecognized configuration class <class 'transformers.models.deberta_v2.configuration_deberta_v2.DebertaV2Config'> for this kind of AutoModel: AutoModelForCausalLM.
Model type should be one of ArceeConfig, AriaTextConfig, BambaConfig

특수 토큰 설정:
  - pad_token: [PAD] (id: 0)
  - bos_token: [CLS] (id: 2)
  - eos_token: None (id: None)
EncoderDecoder 모델 생성 중...


Some weights of ElectraForCausalLM were not initialized from the model checkpoint at monologg/koelectra-small-v2-discriminator and are newly initialized: ['electra.encoder.layer.0.crossattention.output.LayerNorm.bias', 'electra.encoder.layer.0.crossattention.output.LayerNorm.weight', 'electra.encoder.layer.0.crossattention.output.dense.bias', 'electra.encoder.layer.0.crossattention.output.dense.weight', 'electra.encoder.layer.0.crossattention.self.key.bias', 'electra.encoder.layer.0.crossattention.self.key.weight', 'electra.encoder.layer.0.crossattention.self.query.bias', 'electra.encoder.layer.0.crossattention.self.query.weight', 'electra.encoder.layer.0.crossattention.self.value.bias', 'electra.encoder.layer.0.crossattention.self.value.weight', 'electra.encoder.layer.1.crossattention.output.LayerNorm.bias', 'electra.encoder.layer.1.crossattention.output.LayerNorm.weight', 'electra.encoder.layer.1.crossattention.output.dense.bias', 'electra.encoder.layer.1.crossattention.output.dense.

모델 설정:
  - decoder_start_token_id: 2
  - bos_token_id: 2
  - eos_token_id: None
  - pad_token_id: 0
모델 Fine-tuning 중... (간단한 학습)


...

모델 설정:
  - decoder_start_token_id: 1
  - bos_token_id: 1
  - eos_token_id: None
  - pad_token_id: 3
모델 Fine-tuning 중... (간단한 학습)


Training:   0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:557: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWar

Fine-tuning 완료! 평가 시작...


Evaluating beomi/KcELECTRA-base-encoder-decoder:   0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:1739: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
Evaluating beomi/KcELECTRA-base-encoder-decoder: 100%|██████████| 100/100 [04:03<00:00,  2.44s/it]

✓ 결과가 저장되었습니다: /content/drive/MyDrive/data/open/model_evaluation_results.csv

[beomi/KcELECTRA-base-encoder-decoder] 평가 결과:
  - 문자 정확도: 0.0166
  - 단어 정확도: 0.0000
  - 유사도: 0.0237
  - 평균 점수: 0.0134


In [ ]:
# 결과 파일 경로
results_file_path = '/content/drive/MyDrive/data/open/model_evaluation_results.csv'

# 기존 결과 파일이 있으면 로드
import os
if os.path.exists(results_file_path):
    print("\n기존 평가 결과 파일을 찾았습니다. 결과를 추가합니다...")
    existing_results = pd.read_csv(results_file_path, encoding='utf-8-sig')

    # 새로운 결과를 데이터프레임으로 변환
    new_results_df = pd.DataFrame(results)

    # 기존 결과와 병합 (중복 모델은 새 결과로 업데이트)
    combined_results = pd.concat([existing_results, new_results_df], ignore_index=True)
    combined_results = combined_results.drop_duplicates(subset=['model_name'], keep='last')
    results_df = combined_results.sort_values('avg_score', ascending=False).reset_index(drop=True)
else:
    print("\n새로운 평가 결과 파일을 생성합니다...")
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('avg_score', ascending=False).reset_index(drop=True)


기존 평가 결과 파일을 찾았습니다. 결과를 추가합니다...


In [ ]:
print("\n" + "="*60)
print("최종 평가 결과 (성능 순)")
print("="*60)
print(results_df.to_string(index=False))


최종 평가 결과 (성능 순)
                                               model_name  char_accuracy  word_accuracy  similarity  avg_score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [ ]:
# 결과 저장
results_df.to_csv(results_file_path, index=False, encoding='utf-8-sig')
print(f"\n평가 결과가 '{results_file_path}'에 저장되었습니다.")
print(f"총 {len(results_df)}개의 모델이 평가되었습니다.")


평가 결과가 '/content/drive/MyDrive/data/open/model_evaluation_results.csv'에 저장되었습니다.
총 19개의 모델이 평가되었습니다.
